In [1]:
import gzip
import pandas as pd
from collections import defaultdict
from google.colab import drive
drive.mount('/content/gdrive')
import os
os.chdir(r"/content/gdrive/MyDrive/Colab Notebooks")
#os.chdir('/content/driver/MyDrive')
cwd = os.getcwd()  # Get the current working directory (cwd)
files = os.listdir(cwd)  # Get all the files in that directory
print("Files in %r: %s" % (cwd, files))

Mounted at /content/gdrive
Files in '/content/gdrive/MyDrive/Colab Notebooks': ['glove.6B.zip.2', 'glove.6B.zip.1', 'glove.6B.zip', 'SMSSpamCollection.csv', 'Untitled0.ipynb', 'Copy of A Visual Notebook to Using BERT for the First Time.ipynb', 'Untitled1.ipynb', 'hw2.ipynb', 'train_json.gz', 'pairs_Rating.txt', 'pairs_Purchase.txt', 'test_Category.json', 'test_Category_json.gz', 'predictions_Rating.txt', 'predictions_Purchase1.txt', 'ML2_Final.ipynb', 'predictions_Purchase.txt', 'predictions_Category.txt', 'ML_2_FINAL_1.ipynb', 'glove.6B.50d.txt', 'glove.6B.100d.txt', 'glove.6B.200d.txt', 'glove.6B.300d.txt', 'classifcation_reviews.csv', 'HK_Classification', 'HK_Multi_Classification', 'LSTM_category_predictions.csv', 'HK_Multi_Classification_ngram', 'HK_Multi_Classification_LSTM.ipynb']


In [2]:
def readGz(filename):
  f = gzip.open(filename, 'rb')
  for l in (f):
    yield eval(l)

In [3]:
reviews = []
for l in readGz("train_json.gz"):
    reviews.append(l)
print('Total Number of Reviews: ' , (len(reviews)))

df_reviews = pd.DataFrame (reviews)
#df_reviews.head()

Total Number of Reviews:  200000


In [4]:
%pip install contractions

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.util import ngrams
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn import metrics
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
import keras
from keras.layers import Dense, Embedding, LSTM, Dropout
from keras.models import Sequential
from keras.preprocessing.text import Tokenizer
from nltk.stem import WordNetLemmatizer
import re
import collections
import contractions
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
nltk.download('wordnet')
nltk.download('omw-1.4')

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 106 kB 39.3 MB/s 
     |████████████████████████████████| 287 kB 61.7 MB/s 


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [5]:
stop_words = stopwords.words('english')
stemmer = SnowballStemmer('english')

text_cleaning_re = "@\S+|https?:\S+|http?:\S|[^A-Za-z0-9]+"

In [6]:
def preprocess(text, stem=False):
  text = re.sub(text_cleaning_re, ' ', str(text).lower()).strip()
  tokens = []
  for token in text.split():
    if token not in stop_words:
      if stem:
        tokens.append(stemmer.stem(token))
      else:
        tokens.append(token)
  return " ".join(tokens)

In [7]:
df_reviews.reviewText = df_reviews.reviewText.apply(lambda x: preprocess(x))

In [8]:
TRAIN_SIZE = 0.8
train_data, test_data = train_test_split(df_reviews, test_size=1-TRAIN_SIZE,random_state=7)
print("Train Data size:", len(train_data))
print("Test Data size", len(test_data))

Train Data size: 160000
Test Data size 40000


In [9]:
#Tokenization into words
from keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_data.reviewText)

word_index = tokenizer.word_index
vocab_size = len(tokenizer.word_index) + 1
print("Vocabulary Size :", vocab_size)

Vocabulary Size : 56756


In [10]:
from keras_preprocessing.sequence import pad_sequences
MAX_SEQUENCE_LENGTH = 200

x_train = pad_sequences(tokenizer.texts_to_sequences(train_data.reviewText),
                        maxlen = MAX_SEQUENCE_LENGTH)
x_test = pad_sequences(tokenizer.texts_to_sequences(test_data.reviewText),
                       maxlen = MAX_SEQUENCE_LENGTH)


print("Training X Shape:",x_train.shape)
print("Testing X Shape:",x_test.shape)

Training X Shape: (160000, 200)
Testing X Shape: (40000, 200)


In [19]:
# encode class values as integers
from sklearn.preprocessing import LabelEncoder
#from tensorflow.keras.utils import to_categorical
from keras.utils import to_categorical
#import tensorflow.keras.utils.np_utils
encoder = LabelEncoder()
Y_train = train_data['categoryID']
encoder.fit(Y_train)
encoded_Y_train = encoder.transform(Y_train)
dummy_Y_train = to_categorical(encoded_Y_train)

Y_test = test_data['categoryID']
encoder.fit(Y_test)
encoded_Y_test = encoder.transform(Y_test)
dummy_Y_test = to_categorical(encoded_Y_test)

# convert integers to dummy variables (i.e. one hot encoded)

In [20]:
count_0  = 0
count_1 = 0 
count_2 = 0 
count_3 = 0 
count_4 = 0

for i in range(len(dummy_Y_test)):
  if dummy_Y_test[i][0] == 1:
    count_0 = count_0 + 1
  if dummy_Y_test[i][1] == 1:
    count_1 = count_1 + 1
  if dummy_Y_test[i][2] == 1:
    count_2 = count_2 + 1
  if dummy_Y_test[i][3] == 1:
    count_3 = count_3 + 1
  if dummy_Y_test[i][4] == 1:
    count_4 = count_4 + 1

print(count_0,count_1,count_2,count_3,count_4)

28294 10230 511 359 606


In [21]:
import numpy as np
l = len(dummy_Y_train)

count_0  = 0
count_1 = 0 
count_2 = 0 
count_3 = 0 
count_4 = 0

for i in range(len(dummy_Y_train)):
  if dummy_Y_train[i][0] == 1:
    count_0 = count_0 + 1
  if dummy_Y_train[i][1] == 1:
    count_1 = count_1 + 1
  if dummy_Y_train[i][2] == 1:
    count_2 = count_2 + 1
  if dummy_Y_train[i][3] == 1:
    count_3 = count_3 + 1
  if dummy_Y_train[i][4] == 1:
    count_4 = count_4 + 1

print(count_0,count_1,count_2,count_3,count_4)

113104 41186 1818 1522 2370


In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zipAa


--2022-11-23 17:56:41--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2022-11-23 17:56:41--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2022-11-23 17:56:41--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip.2’

gl

In [22]:
GLOVE_EMB = 'glove.6B.300d.txt'
EMBEDDING_DIM = 300
LR = 1e-3
BATCH_SIZE = 1024
EPOCHS = 10
MODEL_PATH = '.../output/kaggle/working/best_model.hdf5'

In [23]:
import numpy as np
embeddings_index = {}

f = open(GLOVE_EMB)
for line in f:
  values = line.split()
  word = value = values[0]
  coefs = np.asarray(values[1:], dtype='float32')
  embeddings_index[word] = coefs
f.close()

print('Found %s word vectors.' %len(embeddings_index))

Found 400000 word vectors.


In [24]:
embedding_matrix = np.zeros((vocab_size, EMBEDDING_DIM))
for word, i in word_index.items():
  embedding_vector = embeddings_index.get(word)
  if embedding_vector is not None:
    embedding_matrix[i] = embedding_vector

In [25]:
import tensorflow as tf
embedding_layer = tf.keras.layers.Embedding(vocab_size,
                                          EMBEDDING_DIM,
                                          weights=[embedding_matrix],
                                          input_length=MAX_SEQUENCE_LENGTH,
                                          trainable=False)

In [26]:
from tensorflow.keras.layers import Conv1D, Bidirectional, LSTM, Dense, Input, Dropout
from tensorflow.keras.layers import SpatialDropout1D
from tensorflow.keras.callbacks import ModelCheckpoint

In [27]:
from keras.layers import Dropout
from tensorflow.keras.layers import SpatialDropout1D
model = Sequential()
model.add(Embedding(MAX_SEQUENCE_LENGTH, EMBEDDING_DIM, input_length=x_train.shape[1]))
model.add(SpatialDropout1D(0.2))
model.add(LSTM(100, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(5, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


In [28]:
print(x_train.shape)
print(x_train.shape[1])
#encoded_Y_train.shape

(160000, 200)
200


In [29]:
# define baseline model
from keras.wrappers.scikit_learn import KerasClassifier
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from keras.callbacks import EarlyStopping
#def baseline_model():
	# create model
model = Sequential()
model.add(Dense(8, input_dim=30,activation='relu'))
#model.add(LSTM(8, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(5, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])



In [30]:
model = Sequential()
model.add(Embedding(MAX_SEQUENCE_LENGTH, EMBEDDING_DIM, input_length=x_train.shape[1]))
model.add(SpatialDropout1D(0.7))
model.add(LSTM(64, dropout=0.7, recurrent_dropout=0.7))
model.add(Dense(5, activation='softmax'))
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['acc'])
print(model.summary())

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_2 (Embedding)     (None, 200, 300)          60000     
                                                                 
 spatial_dropout1d_1 (Spatia  (None, 200, 300)         0         
 lDropout1D)                                                     
                                                                 
 lstm_1 (LSTM)               (None, 64)                93440     
                                                                 
 dense_3 (Dense)             (None, 5)                 325       
                                                                 
Total params: 153,765
Trainable params: 153,765
Non-trainable params: 0
_________________________________________________________________
None


In [31]:
#model.fit(x_train,dummy_Y_train, epochs=5, verbose=0)
epochs = 2
batch_size = 128
history = model.fit(x_train,dummy_Y_train, epochs=epochs, batch_size=batch_size,validation_split=0.2,callbacks=[EarlyStopping(monitor='val_loss',patience=7, min_delta=0.0001)])

Epoch 1/2
1000/1000 [==============================] - 373s 369ms/step - loss: 0.6921 - acc: 0.7264 - val_loss: 0.6408 - val_acc: 0.7480
Epoch 2/2
1000/1000 [==============================] - 372s 372ms/step - loss: 0.6319 - acc: 0.7470 - val_loss: 0.6155 - val_acc: 0.7522


In [32]:
y_pred = model.predict(x_test)

1250/1250 [==============================] - 63s 50ms/step


In [33]:
count_0 = 0
count_1 = 0
count_2 = 0
count_3 = 0
count_4 = 0


for i in range(len(y_pred)):
  if y_pred[i][0] > 0.5:
     count_0 = count_0 + 1
  
  if y_pred[i][1] > 0.5:
     count_1 = count_1 + 1

  if y_pred[i][2] > 0.5:
     count_2 = count_2 + 1
  
  if y_pred[i][3] > 0.5:
     count_3 = count_3 + 1

  if y_pred[i][4] > 0.5:
     count_3 = count_3 + 1

print (count_0,count_1,count_2,count_3,count_4 )

#Y_TEST 28208 10334 474 391 593

33734 5195 0 0 0


In [35]:
from tensorflow.keras import backend as K

def f1(y_true, y_pred):    
    def recall_m(y_true, y_pred):
        TP = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
        Positives = K.sum(K.round(K.clip(y_true, 0, 1)))
        
        recall = TP / (Positives+K.epsilon())    
        return recall 
    
    
    def precision_m(y_true, y_pred):
        TP = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
        Pred_Positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    
        precision = TP / (Pred_Positives+K.epsilon())
        return precision 
    
    precision, recall = precision_m(y_true, y_pred), recall_m(y_true, y_pred)
    
    return 2*((precision*recall)/(precision+recall+K.epsilon()))

def precision(y_true, y_pred,class_to_analyse):
     pred = K.argmax(y_pred)
     true = K.argmax(y_true)
     p = K.cast(K.equal(pred,class_to_analyse),'float64')
     t = K.cast(K.equal(true,class_to_analyse),'float64')
     # Compute the true positive
     common = K.sum(K.dot(K.reshape(t,(1,-1)),K.reshape(p,(-1,1))))
     # divide by all positives in t
     precision = common/ (K.sum(p) + K.epsilon())
     return precision

acc = f1(dummy_Y_test, y_pred)

print(acc)

for i in range(5):
  pr = precision(dummy_Y_test, y_pred,i)
  print('precision for class', i,  pr)



tf.Tensor(0.7536393, shape=(), dtype=float32)
precision for class 0 tf.Tensor(0.7728982591176832, shape=(), dtype=float64)
precision for class 1 tf.Tensor(0.6512902085427485, shape=(), dtype=float64)
precision for class 2 tf.Tensor(0.0, shape=(), dtype=float64)
precision for class 3 tf.Tensor(0.39999999200000014, shape=(), dtype=float64)
precision for class 4 tf.Tensor(0.25581395289345593, shape=(), dtype=float64)


In [36]:
#preidct categories for test data using predictions from the Model

In [37]:
reviews_test = []
for l in readGz("test_Category_json.gz"):
    reviews_test.append(l)
print('Total Number of Reviews: ' , (len(reviews)))

df_reviews_test = pd.DataFrame (reviews_test)
df_reviews_test.head()

Total Number of Reviews:  200000


,reviewTime,reviewText,helpful,reviewerID,reviewHash,unixReviewTime,rating,summary,price
0,"07 26, 2013","I love this blouse, in fact I have it on right...","{'nHelpful': 9, 'outOf': 9}",U281659737,R934811302,1374796800,5.0,love it,NaN
1,"06 8, 2014",Cute product. Loved the fit. Fast shipping! I ...,"{'nHelpful': 0, 'outOf': 0}",U670561057,R657711680,1402185600,4.0,Cute but a bit uncomfortable,NaN
2,"03 21, 2012",I wanted a formal watch that had a big face an...,"{'nHelpful': 0, 'outOf': 0}",U433746872,R750304163,1332288000,5.0,"Bold, Large-face Watch",NaN
3,"06 27, 2014",My daughter used this dress for her first comm...,"{'nHelpful': 0, 'outOf': 0}",U327816997,R865011815,1403827200,5.0,The perfect Dress,NaN
4,"03 21, 2014","Nice shirt, good quality with pockets. Kind o...","{'nHelpful': 1, 'outOf': 1}",U323131234,R222729968,1395360000,5.0,Nice shirt,NaN


In [38]:
df_reviews_test.reviewText = df_reviews_test.reviewText.apply(lambda x: preprocess(x))

In [39]:
df_reviews_test.head()

,reviewTime,reviewText,helpful,reviewerID,reviewHash,unixReviewTime,rating,summary,price
0,"07 26, 2013",love blouse fact right friends like want one,"{'nHelpful': 9, 'outOf': 9}",U281659737,R934811302,1374796800,5.0,love it,NaN
1,"06 8, 2014",cute product loved fit fast shipping would rec...,"{'nHelpful': 0, 'outOf': 0}",U670561057,R657711680,1402185600,4.0,Cute but a bit uncomfortable,NaN
2,"03 21, 2012",wanted formal watch big face definitely cooles...,"{'nHelpful': 0, 'outOf': 0}",U433746872,R750304163,1332288000,5.0,"Bold, Large-face Watch",NaN
3,"06 27, 2014",daughter used dress first communion fit well l...,"{'nHelpful': 0, 'outOf': 0}",U327816997,R865011815,1403827200,5.0,The perfect Dress,NaN
4,"03 21, 2014",nice shirt good quality pockets kind heavyweig...,"{'nHelpful': 1, 'outOf': 1}",U323131234,R222729968,1395360000,5.0,Nice shirt,NaN


In [40]:
df_reviews_test.shape

(14000, 9)

In [41]:
#train_data_x, test_data_x = train_test_split(df_reviews_test, test_size=0.1,random_state=7)
test_data_x = df_reviews_test['reviewText'].to_numpy()

In [42]:
test_data_x.shape

(14000,)

In [43]:
x_test = pad_sequences(tokenizer.texts_to_sequences(test_data_x),
                       maxlen = MAX_SEQUENCE_LENGTH)
#y_test = encoder.transform(df_reviews_test.categoryID.to_list())


In [44]:
x_test.shape

(14000, 200)

In [45]:
predictions_x = model.predict(x_test)
print(len(predictions_x))


438/438 [==============================] - 22s 49ms/step
14000


In [46]:
print(len(predictions_x[0]))
pred_t = predictions_x.T

5


In [47]:
df_reviews_test['prediction_0'] = pred_t[0]
df_reviews_test['prediction_1'] = pred_t[1]
df_reviews_test['prediction_2'] = pred_t[2]
df_reviews_test['prediction_3'] = pred_t[3]
df_reviews_test['prediction_4'] = pred_t[4]

In [48]:
df_reviews_test.head()

,reviewTime,reviewText,helpful,reviewerID,reviewHash,unixReviewTime,rating,summary,price,prediction_0,prediction_1,prediction_2,prediction_3,prediction_4
0,"07 26, 2013",love blouse fact right friends like want one,"{'nHelpful': 9, 'outOf': 9}",U281659737,R934811302,1374796800,5.0,love it,NaN,0.878054,0.109203,0.006101,0.002461,0.004181
1,"06 8, 2014",cute product loved fit fast shipping would rec...,"{'nHelpful': 0, 'outOf': 0}",U670561057,R657711680,1402185600,4.0,Cute but a bit uncomfortable,NaN,0.949682,0.025652,0.010488,0.002700,0.011479
2,"03 21, 2012",wanted formal watch big face definitely cooles...,"{'nHelpful': 0, 'outOf': 0}",U433746872,R750304163,1332288000,5.0,"Bold, Large-face Watch",NaN,0.262883,0.728504,0.001648,0.004078,0.002886
3,"06 27, 2014",daughter used dress first communion fit well l...,"{'nHelpful': 0, 'outOf': 0}",U327816997,R865011815,1403827200,5.0,The perfect Dress,NaN,0.972114,0.019063,0.005194,0.000851,0.002779
4,"03 21, 2014",nice shirt good quality pockets kind heavyweig...,"{'nHelpful': 1, 'outOf': 1}",U323131234,R222729968,1395360000,5.0,Nice shirt,NaN,0.407006,0.570011,0.005333,0.009325,0.008325


In [49]:
#write it to a file to save time
df_reviews_test.to_csv('LSTM_category_predictions.csv')